In [9]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------------
# Sample: the 21 complete SSA countries, minus the two stale-year outliers
# (Cabo Verde 2009, Sierra Leone 2014 — dropped per prior decision)
# ---------------------------------------------------------------
COMPLETE_19 = {
    "AGO": "Angola", "BWA": "Botswana", "BDI": "Burundi",
    "SWZ": "Eswatini", "ETH": "Ethiopia", "GMB": "Gambia, The", "GHA": "Ghana",
    "GNB": "Guinea-Bissau", "LSO": "Lesotho", "MWI": "Malawi", "MLI": "Mali",
    "NGA": "Nigeria", "RWA": "Rwanda", "SEN": "Senegal", "SYC": "Seychelles",
    "SDN": "Sudan", "TZA": "Tanzania", "UGA": "Uganda", "ZMB": "Zambia",
}

master = pd.DataFrame({"iso3": list(COMPLETE_19.keys()), "country": list(COMPLETE_19.values())})

# ---------------------------------------------------------------
# GAI exposure: total_exposure_share = (GRADIENT_TOTAL - GRADIENT_X) / GRADIENT_TOTAL
# computed separately for SEX_T, SEX_F, SEX_M, using each country's latest available year
# ---------------------------------------------------------------
gai = pd.read_csv("EMP_TEMP_SEX_GAI_NB_A-20260911T1441.csv.gz")
gai_19 = gai[gai["ref_area"].isin(COMPLETE_19.keys())].copy()

latest_year = (
    gai_19[(gai_19["classif1"] == "GAI_GRADIENT_TOTAL") & (gai_19["sex"] == "SEX_T")]
    .dropna(subset=["obs_value"])
    .groupby("ref_area")["time"].max()
)
gai_19 = gai_19[gai_19.apply(lambda r: r["time"] == latest_year.get(r["ref_area"], -1), axis=1)]

rows = []
for iso3, grp in gai_19.groupby("ref_area"):
    row = {"iso3": iso3, "exposure_year": latest_year[iso3]}
    for sex_code, label in [("SEX_T", "total"), ("SEX_F", "female"), ("SEX_M", "male")]:
        sub = grp[grp["sex"] == sex_code]
        total = sub.loc[sub["classif1"] == "GAI_GRADIENT_TOTAL", "obs_value"]
        x = sub.loc[sub["classif1"] == "GAI_GRADIENT_X", "obs_value"]
        row[f"{label}_exposure_share"] = (
            1 - (x.values[0] / total.values[0]) if len(total) and len(x) and total.values[0] > 0 else np.nan
        )
    rows.append(row)

exposure_df = pd.DataFrame(rows)
master = master.merge(exposure_df, on="iso3", how="left")
master["gender_gap"] = master["female_exposure_share"] - master["male_exposure_share"]

# ---------------------------------------------------------------
# AIPI overall score
# ---------------------------------------------------------------
aipi = pd.read_csv("data/raw/imf_aipi/aipi_overall_aipi.csv")[["iso3", "AI_PI"]]
master = master.merge(aipi[aipi["iso3"].isin(COMPLETE_19.keys())], on="iso3", how="left")
# ---------------------------------------------------------------
# Standardize within-sample, compute the Adjustment Gap
# ---------------------------------------------------------------
zscore = lambda s: (s - s.mean()) / s.std(ddof=0)
master["z_exposure"] = zscore(master["total_exposure_share"])
master["z_aipi"] = zscore(master["AI_PI"])
master["adjustment_gap"] = master["z_exposure"] - master["z_aipi"]

# ---------------------------------------------------------------
# Quadrant typology
# ---------------------------------------------------------------
def quadrant(row):
    high_exp, high_cap = row["z_exposure"] >= 0, row["z_aipi"] >= 0
    if high_exp and not high_cap:  return "High exposure / Low capacity (highest concern)"
    if high_exp and high_cap:      return "High exposure / High capacity (well-matched)"
    if not high_exp and high_cap:  return "Low exposure / High capacity (well-matched)"
    return "Low exposure / Low capacity"

master["quadrant"] = master.apply(quadrant, axis=1)
master = master.sort_values("adjustment_gap", ascending=False).reset_index(drop=True)
master.insert(0, "rank", range(1, len(master) + 1))

master.to_csv("ssa_adjustment_gap.csv", index=False)
print(master[["rank", "iso3", "country", "exposure_year", "total_exposure_share",
              "gender_gap", "AI_PI", "adjustment_gap", "quadrant"]].to_string(index=False))

 rank iso3       country  exposure_year  total_exposure_share  gender_gap    AI_PI  adjustment_gap                                       quadrant
    1  AGO        Angola           2025              0.270693    0.084708 0.259659        1.798795 High exposure / Low capacity (highest concern)
    2  SDN         Sudan           2022              0.224421   -0.058364 0.232907        1.727928 High exposure / Low capacity (highest concern)
    3  SWZ      Eswatini           2023              0.312732   -0.021061 0.306211        1.554809 High exposure / Low capacity (highest concern)
    4  GMB   Gambia, The           2025              0.353390    0.057297 0.359976        1.197647   High exposure / High capacity (well-matched)
    5  MWI        Malawi           2024              0.315721    0.048673 0.340047        1.114317 High exposure / Low capacity (highest concern)
    6  GNB Guinea-Bissau           2022              0.177749    0.005711 0.264899        0.838969                    Low ex

In [15]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------------
# Sample: 21 complete SSA countries, minus Cabo Verde/Sierra Leone
# (already excluded for stale years), with exposure_year >= 2022
# enforced as a standing rule from this point on.
# ---------------------------------------------------------------
COMPLETE_19 = {
    "AGO": "Angola", "BWA": "Botswana", "BDI": "Burundi",
    "SWZ": "Eswatini", "ETH": "Ethiopia", "GMB": "Gambia, The", "GHA": "Ghana",
    "GNB": "Guinea-Bissau", "LSO": "Lesotho", "MWI": "Malawi", "MLI": "Mali",
    "NGA": "Nigeria", "RWA": "Rwanda", "SEN": "Senegal", "SYC": "Seychelles",
    "SDN": "Sudan", "TZA": "Tanzania", "UGA": "Uganda", "ZMB": "Zambia",
}

master = pd.DataFrame({"iso3": list(COMPLETE_19.keys()), "country": list(COMPLETE_19.values())})

# ---------------------------------------------------------------
# GAI exposure: total/female/male shares, using each country's
# latest available year (same logic as before)
# ---------------------------------------------------------------
gai = pd.read_csv("EMP_TEMP_SEX_GAI_NB_A-20260911T1441.csv.gz")
gai_19 = gai[gai["ref_area"].isin(COMPLETE_19.keys())].copy()

latest_year = (
    gai_19[(gai_19["classif1"] == "GAI_GRADIENT_TOTAL") & (gai_19["sex"] == "SEX_T")]
    .dropna(subset=["obs_value"])
    .groupby("ref_area")["time"].max()
)
gai_19 = gai_19[gai_19.apply(lambda r: r["time"] == latest_year.get(r["ref_area"], -1), axis=1)]

rows = []
for iso3, grp in gai_19.groupby("ref_area"):
    row = {"iso3": iso3, "exposure_year": latest_year[iso3]}
    for sex_code, label in [("SEX_T", "total"), ("SEX_F", "female"), ("SEX_M", "male")]:
        sub = grp[grp["sex"] == sex_code]
        total = sub.loc[sub["classif1"] == "GAI_GRADIENT_TOTAL", "obs_value"]
        x = sub.loc[sub["classif1"] == "GAI_GRADIENT_X", "obs_value"]
        row[f"{label}_exposure_share"] = (
            1 - (x.values[0] / total.values[0]) if len(total) and len(x) and total.values[0] > 0 else np.nan
        )
    rows.append(row)

exposure_df = pd.DataFrame(rows)
master = master.merge(exposure_df, on="iso3", how="left")
master["gender_gap"] = master["female_exposure_share"] - master["male_exposure_share"]

# ---------------------------------------------------------------
# NEW: enforce the 2022+ rule, drop the stale-year countries
# ---------------------------------------------------------------
dropped = master.loc[master["exposure_year"] < 2022, ["iso3", "country", "exposure_year"]]
print(f"Dropping {len(dropped)} countries for exposure_year < 2022:")
print(dropped.to_string(index=False))

master = master[master["exposure_year"] >= 2022].reset_index(drop=True)
print(f"\nWorking sample: {len(master)} countries\n")

# ---------------------------------------------------------------
# AIPI overall score
# ---------------------------------------------------------------
aipi = pd.read_csv("data/raw/imf_aipi/aipi_overall_aipi.csv")[["iso3", "AI_PI"]]
master = master.merge(aipi[aipi["iso3"].isin(master["iso3"])], on="iso3", how="left")

# ---------------------------------------------------------------
# Standardize WITHIN this now-smaller sample, compute the gap
# ---------------------------------------------------------------
zscore = lambda s: (s - s.mean()) / s.std(ddof=0)
master["z_exposure"] = zscore(master["total_exposure_share"])
master["z_aipi"] = zscore(master["AI_PI"])
master["adjustment_gap"] = master["z_exposure"] - master["z_aipi"]

def quadrant(row):
    high_exp, high_cap = row["z_exposure"] >= 0, row["z_aipi"] >= 0
    if high_exp and not high_cap:  return "High exposure / Low capacity (highest concern)"
    if high_exp and high_cap:      return "High exposure / High capacity (well-matched)"
    if not high_exp and high_cap:  return "Low exposure / High capacity (well-matched)"
    return "Low exposure / Low capacity"

master["quadrant"] = master.apply(quadrant, axis=1)
master = master.sort_values("adjustment_gap", ascending=False).reset_index(drop=True)
master.insert(0, "rank", range(1, len(master) + 1))

master.to_csv("ssa_adjustment_gap.csv", index=False)
print(master[["rank", "iso3", "country", "exposure_year", "total_exposure_share",
              "female_exposure_share", "male_exposure_share", "gender_gap",
              "AI_PI", "z_exposure", "z_aipi", "adjustment_gap", "quadrant"]].to_string(index=False))

Dropping 6 countries for exposure_year < 2022:
iso3    country  exposure_year
 BDI    Burundi           2020
 ETH   Ethiopia           2021
 RWA     Rwanda           2021
 SYC Seychelles           2020
 TZA   Tanzania           2020
 UGA     Uganda           2021

Working sample: 13 countries

 rank iso3       country  exposure_year  total_exposure_share  female_exposure_share  male_exposure_share  gender_gap    AI_PI  z_exposure    z_aipi  adjustment_gap                                       quadrant
    1  AGO        Angola           2025              0.270693               0.312092             0.227384    0.084708 0.259659    0.505880 -1.302473        1.808352 High exposure / Low capacity (highest concern)
    2  SDN         Sudan           2022              0.224421               0.176482             0.234847   -0.058364 0.232907   -0.063474 -1.764386        1.700912                    Low exposure / Low capacity
    3  SWZ      Eswatini           2023              0.312732        

In [18]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

output_dir = Path.cwd() / "figures"
output_dir.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans"],
    "font.size": 11,
    "axes.edgecolor": "#444444",
    "axes.linewidth": 0.8,
})

data = [
    # country, z_exposure, z_aipi, gap
    ("Angola",        0.51, -1.30, 1.81),
    ("Sudan",        -0.06, -1.76, 1.70),
    ("Eswatini",      1.02, -0.50, 1.52),
    ("Gambia",        1.52,  0.43, 1.09),
    ("Malawi",        1.06,  0.09, 0.97),
    ("Guinea-Bissau", -0.64, -1.21, 0.57),
    ("Nigeria",       0.29,  0.02, 0.27),
    ("Mali",         -1.09, -0.67,-0.42),
    ("Botswana",      0.52,  1.34,-0.82),
    ("Ghana",         0.65,  1.56,-0.91),
    ("Lesotho",      -0.89,  0.35,-1.24),
    ("Zambia",       -0.78,  0.62,-1.39),
    ("Senegal",      -2.11,  1.05,-3.16),
]

# Category by which z-score is more extreme (dominant driver) x sign of gap
CAT_COLORS = {
    "Concern — capacity-driven":  "#B33F3F",
    "Concern — exposure-driven":  "#E08A2E",
    "Reassuring — capacity-driven": "#3F7CB3",
    "Reassuring — exposure-driven": "#5FA8A0",
}

def categorize(ze, za, gap):
    concern = gap > 0
    dominant = "capacity-driven" if abs(za) > abs(ze) else "exposure-driven"
    prefix = "Concern" if concern else "Reassuring"
    return f"{prefix} — {dominant}"

for i, row in enumerate(data):
    country, ze, za, gap = row
    data[i] = (country, ze, za, gap, categorize(ze, za, gap))

# ------------------------------------------------------------------
# FIGURE 1: scatter of exposure z vs capacity z
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7.2, 6.2), dpi=300)

lim = 2.4
ax.axhline(0, color="#999999", linewidth=0.9, zorder=1)
ax.axvline(0, color="#999999", linewidth=0.9, zorder=1)
ax.plot([-lim, lim], [-lim, lim], color="#999999", linewidth=0.9, linestyle="--", zorder=1)

# quadrant labels
ax.text(-lim+0.08, lim-0.15, "High exposure / Low capacity", fontsize=8.5, color="#777777", va="top", ha="left", style="italic")
ax.text(lim-0.08, lim-0.15, "High exposure / High capacity", fontsize=8.5, color="#777777", va="top", ha="right", style="italic")
ax.text(-lim+0.08, -lim+0.1, "Low exposure / Low capacity", fontsize=8.5, color="#777777", va="bottom", ha="left", style="italic")
ax.text(lim-0.08, -lim+0.1, "Low exposure / High capacity", fontsize=8.5, color="#777777", va="bottom", ha="right", style="italic")

for country, ze, za, gap, cat in data:
    ax.scatter(za, ze, s=90, color=CAT_COLORS[cat], edgecolor="white", linewidth=0.8, zorder=3)
    dx, dy = 0.06, 0.06
    ha = "left"
    if country in ("Ghana",):
        dy = -0.16
    if country in ("Nigeria",):
        dy = -0.02
        dx = -0.10
        ha = "right"
    ax.annotate(country, (za, ze), xytext=(za+dx, ze+dy), fontsize=9, ha=ha, zorder=4)

ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_xlabel("AIPI z-score (adaptive capacity, relative to sample)")
ax.set_ylabel("Exposure z-score (relative to sample)")

handles = [mpatches.Patch(color=c, label=l) for l, c in CAT_COLORS.items()]
ax.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, -0.24), ncol=2, fontsize=8.5, frameon=False)

fig.tight_layout()
fig.savefig(output_dir / "figure1_scatter.png", bbox_inches="tight")
fig.savefig(output_dir / "figure1_scatter.pdf", bbox_inches="tight")
plt.close(fig)

# ------------------------------------------------------------------
# FIGURE 2: ranked bar chart, colored by category
# ------------------------------------------------------------------
data_sorted = sorted(data, key=lambda r: r[3], reverse=True)
countries = [r[0] for r in data_sorted]
gaps = [r[3] for r in data_sorted]
cats = [r[4] for r in data_sorted]
colors = [CAT_COLORS[c] for c in cats]

fig2, ax2 = plt.subplots(figsize=(7.2, 5.6), dpi=300)
y_pos = np.arange(len(countries))[::-1]
bars = ax2.barh(y_pos, gaps, color=colors, edgecolor="white", height=0.68, zorder=3)

ax2.axvline(0, color="#444444", linewidth=1.0, zorder=2)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(countries, fontsize=10)
ax2.set_xlabel("Adjustment Gap (z_exposure − z_aipi)")
ax2.grid(axis="x", color="#e5e5e5", linewidth=0.7, zorder=0)
ax2.set_axisbelow(True)

ax2.set_xlim(min(gaps) - 0.75, max(gaps) + 0.55)
for y, g in zip(y_pos, gaps):
    ax2.text(g + (0.08 if g >= 0 else -0.08), y, f"{g:.2f}", va="center",
              ha="left" if g >= 0 else "right", fontsize=8.5, color="#333333")

handles2 = [mpatches.Patch(color=c, label=l) for l, c in CAT_COLORS.items()]
ax2.legend(handles=handles2, loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2, fontsize=8.5, frameon=False)

fig2.tight_layout()
fig2.savefig(output_dir / "figure2_bars.png", bbox_inches="tight")
fig2.savefig(output_dir / "figure2_bars.pdf", bbox_inches="tight")
plt.close(fig2)

print(f"Saved figures to: {output_dir}")
print("- figure1_scatter.png")
print("- figure1_scatter.pdf")
print("- figure2_bars.png")
print("- figure2_bars.pdf")

Saved figures to: c:\Users\HP\Downloads\Social_AI Impact\figures
- figure1_scatter.png
- figure1_scatter.pdf
- figure2_bars.png
- figure2_bars.pdf
